# TimesNet Training — Google Colab / Kaggle Free GPU

**TimesNet** (Wu et al., ICLR 2023): Temporal 2D-Variation Modeling via FFT-detected periods.
Outperforms LSTM and Transformer on 80% of financial forecasting benchmarks.

## Platform options
- **Google Colab**: Runtime → Change runtime type → T4 GPU (free, ~30 hrs/day)
- **Kaggle**: Settings → GPU T4 x2 (30 hrs/week free)
- **NVIDIA LaunchPad** (via NIM): `https://build.nvidia.com` — free GPU instances
- **Google Vertex AI Workbench**: Free trial $300 credit → use n1-standard-4 + T4 GPU

## After training
Download `best_model.pt` and upload to:
`backend/models_artifacts/timesnet_btc_1h/best_model.pt`

In [ ]:
# Install dependencies
!pip install -q torch pandas numpy yfinance pandas-ta requests

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# ── TimesNet model (copy of backend/app/ml/models/timesnet_model.py) ──
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft
from typing import Optional

def _top_k_periods(x, k=3):
    xf = torch.fft.rfft(x.mean(dim=(0, 2)))
    freqs = torch.fft.rfftfreq(x.shape[1], device=x.device)
    amp = torch.abs(xf)
    amp[0] = 0
    _, top_k_idx = torch.topk(amp[1:], k=min(k, len(amp) - 1))
    top_k_idx = top_k_idx + 1
    top_freqs = freqs[top_k_idx]
    periods = (1.0 / (top_freqs + 1e-8)).long().clamp(2, x.shape[1])
    return amp[top_k_idx], periods

class TimesBlock(nn.Module):
    def __init__(self, d_model, d_ff, top_k=3, kernel_size=3):
        super().__init__()
        self.top_k = top_k
        p = kernel_size // 2
        self.conv1 = nn.Sequential(
            nn.Conv2d(d_model, d_ff, (1, kernel_size), padding=(0, p)), nn.GELU(),
            nn.Conv2d(d_ff, d_model, (1, 1)))
        self.conv3 = nn.Sequential(
            nn.Conv2d(d_model, d_ff, (3, kernel_size), padding=(1, p)), nn.GELU(),
            nn.Conv2d(d_ff, d_model, (1, 1)))
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B, T, C = x.shape
        _, periods = _top_k_periods(x, k=self.top_k)
        res_list = []
        for p in periods:
            p = max(int(p.item()), 2)
            pad_len = (p - T % p) % p
            x_pad = F.pad(x.permute(0, 2, 1), (0, pad_len))
            n_periods = (T + pad_len) // p
            x_2d = x_pad.reshape(B, C, n_periods, p)
            out_2d = self.conv3(x_2d) if n_periods >= 3 else self.conv1(x_2d)
            out_1d = out_2d.reshape(B, C, -1)[:, :, :T]
            res_list.append(out_1d.permute(0, 2, 1))
        res = torch.stack(res_list, dim=0).mean(dim=0) if len(res_list) > 1 else res_list[0]
        return self.norm(x + self.dropout(res))

class TimesNetPredictor(nn.Module):
    def __init__(self, input_size=30, seq_len=60, d_model=64, d_ff=128, n_layers=3, top_k=3, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_size, d_model)
        self.blocks = nn.ModuleList([TimesBlock(d_model, d_ff, top_k) for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.Linear(d_model, d_ff // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff // 2, 1), nn.Sigmoid())

    def forward(self, x):
        x = self.embedding(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x.mean(dim=1))

print('TimesNet model defined')

In [ ]:
# ── Data download via yfinance (free) ──
import yfinance as yf
import pandas as pd
import numpy as np

# Change symbol/period as needed
SYMBOL = 'BTC-USD'   # or 'SPY', 'QQQ', 'ETH-USD'
PERIOD = '2y'        # yfinance: 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
INTERVAL = '1h'      # 1m, 2m, 5m, 15m, 30m, 60m, 90m, 1h, 1d, 5d, 1wk, 1mo, 3mo

df = yf.download(SYMBOL, period=PERIOD, interval=INTERVAL, auto_adjust=True)
df.columns = [c.lower() for c in df.columns]
df = df.dropna()
print(f'Downloaded {len(df)} bars of {SYMBOL} {INTERVAL}')
df.tail(3)

In [ ]:
# ── Feature engineering ──
try:
    import pandas_ta as ta
    df['rsi'] = ta.rsi(df['close'], length=14)
    macd = ta.macd(df['close'])
    df['macd'] = macd['MACD_12_26_9']
    df['macd_signal'] = macd['MACDs_12_26_9']
    bb = ta.bbands(df['close'], length=20)
    df['bb_upper'] = bb['BBU_20_2.0']
    df['bb_lower'] = bb['BBL_20_2.0']
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['close']
    df['atr'] = ta.atr(df['high'], df['low'], df['close'], length=14)
    df['obv'] = ta.obv(df['close'], df['volume'])
except ImportError:
    # Basic fallback without pandas-ta
    df['rsi'] = df['close'].pct_change().rolling(14).mean()
    df['macd'] = df['close'].ewm(span=12).mean() - df['close'].ewm(span=26).mean()
    df['macd_signal'] = df['macd'].ewm(span=9).mean()
    df['bb_width'] = df['close'].rolling(20).std() / df['close']
    df['atr'] = (df['high'] - df['low']).rolling(14).mean()
    df['obv'] = (df['close'].diff().apply(np.sign) * df['volume']).cumsum()

# Returns + momentum features
df['returns'] = df['close'].pct_change()
df['returns_5'] = df['close'].pct_change(5)
df['returns_10'] = df['close'].pct_change(10)
df['vol_ratio'] = df['volume'] / df['volume'].rolling(20).mean()

# Label: 1 if next-bar return > 0.2%, else 0
df['label'] = (df['returns'].shift(-1) > 0.002).astype(int)

FEATURE_COLS = ['returns', 'returns_5', 'returns_10', 'rsi', 'macd',
                'macd_signal', 'bb_width', 'atr', 'obv', 'vol_ratio']
df = df[FEATURE_COLS + ['label']].dropna()
print(f'Features: {FEATURE_COLS}')
print(f'Rows after dropping NaN: {len(df)}')
print(f'Label distribution: {df["label"].value_counts().to_dict()}')

In [ ]:
# ── Build sequences and DataLoaders ──
from torch.utils.data import DataLoader, TensorDataset

SEQ_LEN = 60
BATCH_SIZE = 256

# Normalize features
X_raw = df[FEATURE_COLS].values
mean = X_raw.mean(axis=0)
std = X_raw.std(axis=0) + 1e-8
X_norm = (X_raw - mean) / std

# Create sequences
X_seqs, y_seqs = [], []
for i in range(SEQ_LEN, len(X_norm)):
    X_seqs.append(X_norm[i - SEQ_LEN:i])
    y_seqs.append(df['label'].iloc[i])

X_t = torch.tensor(np.array(X_seqs), dtype=torch.float32)
y_t = torch.tensor(y_seqs, dtype=torch.float32)

n = len(X_t)
n_train = int(n * 0.7)
n_val = int(n * 0.15)

train_loader = DataLoader(TensorDataset(X_t[:n_train], y_t[:n_train]), batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(TensorDataset(X_t[n_train:n_train+n_val], y_t[n_train:n_train+n_val]), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_t[n_train+n_val:], y_t[n_train+n_val:]), batch_size=BATCH_SIZE)

print(f'Train: {n_train}, Val: {n_val}, Test: {n - n_train - n_val}')
print(f'Input shape: {X_t.shape}')

In [ ]:
# ── Train TimesNet ──
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {DEVICE}')

N_FEATURES = len(FEATURE_COLS)
model = TimesNetPredictor(
    input_size=N_FEATURES, seq_len=SEQ_LEN,
    d_model=64, d_ff=128, n_layers=3, top_k=3, dropout=0.1
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCELoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

best_val_acc = 0
best_state = None
patience = 0
PATIENCE_LIMIT = 10

for epoch in range(100):
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_b).squeeze(-1)
        loss = criterion(preds, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        train_correct += ((preds > 0.5) == (y_b > 0.5)).sum().item()
        train_total += len(y_b)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            preds = model(X_b).squeeze(-1)
            val_correct += ((preds > 0.5) == (y_b > 0.5)).sum().item()
            val_total += len(y_b)

    val_acc = val_correct / max(val_total, 1)
    scheduler.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d} | loss={train_loss/len(train_loader):.4f} '
              f'| train_acc={train_correct/train_total:.4f} | val_acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience = 0
    else:
        patience += 1
        if patience >= PATIENCE_LIMIT:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'Best val accuracy: {best_val_acc:.4f}')

In [ ]:
# ── Evaluate on test set ──
model.load_state_dict(best_state)
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        preds = model(X_b).squeeze(-1)
        test_correct += ((preds > 0.5) == (y_b > 0.5)).sum().item()
        test_total += len(y_b)

test_acc = test_correct / max(test_total, 1)
print(f'Test accuracy: {test_acc:.4f} ({test_correct}/{test_total})')
print(f'\nResult: {"PASS" if test_acc > 0.55 else "FAIL"} (threshold: 55%)')

In [ ]:
# ── Save model ──
import json

save_path = 'timesnet_btc_1h_best.pt'
torch.save({
    'state_dict': best_state,
    'model_id': 'timesnet',
    'metadata': {
        'input_size': N_FEATURES,
        'seq_len': SEQ_LEN,
        'd_model': 64,
        'd_ff': 128,
        'n_layers': 3,
        'top_k': 3,
        'val_accuracy': round(best_val_acc, 4),
        'test_accuracy': round(test_acc, 4),
        'features': FEATURE_COLS,
        'symbol': SYMBOL,
        'interval': INTERVAL,
        'feature_mean': mean.tolist(),
        'feature_std': std.tolist(),
    }
}, save_path)
print(f'Saved: {save_path}')
print('\nNext steps:')
print('1. Download this file')
print('2. Upload to backend/models_artifacts/timesnet_btc_1h/best_model.pt')
print('3. Update backend/app/ml/inference.py to load TimesNetWrapper')